In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
import numpy as np, sympy as sp
from IPython.display import Image, display, Markdown
sp.init_printing(use_unicode=True)
def show(path):
    p = path[0] if isinstance(path, tuple) else path
    display(Image(filename=p))


# 03. Linearized Einstein equations and Bianchi identities

Two independent derivations of $\delta E_{\mu\nu}=\delta G_{\mu\nu}+\Lambda h_{\mu\nu}$ are compared for a generic polynomial perturbation:
(i) background-covariant formulas and (ii) the exact $O(\epsilon)$ coefficient of $G+\Lambda g$ computed from a truncated inverse metric.
The linearized Bianchi identity is then tested in three forms, including the *naive* divergence, which does not vanish.

In [2]:
from desitter_perturbations.background import symbolic_background
from desitter_perturbations.symbolic.curvature import Geometry
from desitter_perturbations.symbolic import einstein_tensor as et, bianchi_identities as bi
b = symbolic_background(); H = b["H"]; eta = b["eta"]; x, y, z = b["spatial"]
geom = Geometry(b["coords_conformal"], b["g_conformal"])
h = sp.Matrix([[eta*x, y, 0, z*eta],[y, x**2, eta*z, 0],[0, eta*z, y*z, x],[z*eta, 0, x, eta**2]])
cov = et.linearized_einstein_covariant(geom, h, 3*H**2)
e0, e1 = et.linearized_einstein_bruteforce(b["coords_conformal"], b["g_conformal"], h, 3*H**2)
print("background part of the brute-force pipeline (must vanish):", e0 == sp.zeros(4,4))
print("covariant minus component-level dE:", (cov["dE"] - e1).applyfunc(lambda e: sp.cancel(sp.together(e))) == sp.zeros(4,4))

background part of the brute-force pipeline (must vanish): True
covariant minus component-level dE: True


In [3]:
res, naive, corr = bi.bianchi_covariant_residual(geom, cov["dG"], cov["dGamma"])
print("covariant-index Bianchi residual:", res)
print("mixed-index Bianchi residual   :", bi.bianchi_mixed_residual(geom, h, cov["dG"], cov["dGamma"]))
print("nabla^m dE_mn                  :", bi.bianchi_field_equation_residual(geom, cov["dE"]))
print()
print("naive nabla^m dG_mn (NOT zero; proportional to Lambda):")
for n in naive: print("   ", sp.factor(n))

covariant-index Bianchi residual: [0, 0, 0, 0]


mixed-index Bianchi residual   : [0, 0, 0, 0]
nabla^m dE_mn                  : [0, 0, 0, 0]

naive nabla^m dG_mn (NOT zero; proportional to Lambda):
    -3*H**4*eta*(2*eta**2 + x**2 + y*z)
    -6*H**4*eta*(eta*x + y)
    -3*H**4*eta**2*z
    -3*H**4*eta**2*z


The naive divergence is nonzero because $\delta(\nabla^\mu G_{\mu\nu})$ contains $\delta\Gamma\cdot\bar G$ terms and
$\bar G_{\mu\nu}=-\Lambda\bar g_{\mu\nu}\neq0$. For $E=G+\Lambda g$ the background value vanishes, so $\bar\nabla^\mu\delta E_{\mu\nu}=0$ holds exactly.